# P13 - Qualite des donnees et EDA approfondie

Ce notebook couvre :
1. Disponibilite et volumetrie des tables
2. Valeurs manquantes
3. Statistiques descriptives (variables quantitatives)
4. Variables qualitatives - modalites et encodage
5. Distributions et outliers
6. Matrice de correlations features → cibles ML
7. Conclusions et recommandations


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if Path('/app/src').exists():
    sys.path.insert(0, '/app/src')
else:
    sys.path.insert(0, '../../src')

from p13.db import read_sql
from p13.ml.features import ALL_TARGETS, FEATURE_COLUMNS

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


## 1. Volumetrie des tables


In [ ]:
tables = [
    'communes', 'stats_communes', 'ecoles_effectifs',
    'mutations_dvf', 'permis_construire', 'population_2014',
    'referentiel_batiment', 'ml_dataset_commune',
]

vol = []
for t in tables:
    try:
        n = read_sql(f'SELECT COUNT(*) AS n FROM {t}').iloc[0]['n']
        ncols = len(read_sql(f'SELECT * FROM {t} LIMIT 1').columns)
        vol.append({'table': t, 'lignes': int(n), 'colonnes': ncols})
    except Exception as e:
        vol.append({'table': t, 'lignes': -1, 'colonnes': 0})

df_vol = pd.DataFrame(vol)
df_vol


## 2. Valeurs manquantes

Analyse du taux de nulls par colonne sur les tables principales.


In [ ]:
def missing_report(table, limit=500):
    df = read_sql(f'SELECT * FROM {table} LIMIT {limit}')
    miss = (df.isna().sum() / len(df) * 100).round(2)
    miss = miss[miss > 0].sort_values(ascending=False)
    return miss.rename(table)

reports = []
for t in ['ml_dataset_commune', 'stats_communes', 'mutations_dvf']:
    r = missing_report(t)
    if not r.empty:
        reports.append(r)

if reports:
    miss_df = pd.concat(reports, axis=1).fillna(0)
    fig = px.imshow(
        miss_df.T,
        title='Taux de valeurs manquantes (%) par colonne et table',
        color_continuous_scale='Reds',
        labels={'color': '% nulls'}
    )
    fig.show()
    miss_df
else:
    print('Aucune valeur manquante detectee dans les tables analysees.')


## 3. Statistiques descriptives — variables quantitatives

Statistiques classiques sur le dataset ML (features + cibles).


In [ ]:
ml = read_sql('SELECT * FROM ml_dataset_commune')
print(f'Dataset ML : {ml.shape[0]} lignes x {ml.shape[1]} colonnes')
ml.head()


In [ ]:
desc = ml[FEATURE_COLUMNS + ALL_TARGETS].describe().T
desc['cv_%'] = (desc['std'] / desc['mean'].abs() * 100).round(1)
desc = desc.round(2)
desc


**Lecture :**
- `cv_%` (coefficient de variation) indique la dispersion relative : > 100% = feature tres heterogene entre communes.
- Comparer `min` et `max` pour reperer des valeurs aberrantes potentielles.
- Comparer `mean` et `50%` (mediane) : un ecart important signale une distribution asymetrique.


## 4. Variables qualitatives — modalites

Recensement des valeurs distinctes dans les colonnes categoriques.


In [ ]:
dvf = read_sql('SELECT type_bien, periode_construction FROM mutations_dvf LIMIT 5000')
ecoles = read_sql('SELECT secteur FROM ecoles_effectifs LIMIT 5000')
permis = read_sql('SELECT etat_projet, type_habitation FROM permis_construire LIMIT 5000')

print('=== DVF - type_bien ===')
print(dvf['type_bien'].value_counts(dropna=False).head(20))
print()
print('=== DVF - periode_construction ===')
print(dvf['periode_construction'].value_counts(dropna=False))
print()
print('=== Ecoles - secteur (public/prive) ===')
print(ecoles['secteur'].value_counts(dropna=False))
print()
print('=== Permis - etat_projet ===')
print(permis['etat_projet'].value_counts(dropna=False))


**Observations attendues :**
- `secteur` : PUBLIC / PRIVE → variable binaire, peut etre encodee 0/1 pour des analyses specifiques.
- `type_bien` : categorique non ordinale → encodage one-hot si utilise en feature ML.
- `periode_construction` : ordinale (ordre chronologique) → encodage ordinal possible.
- `etat_projet` : filtre qualite (autorisation accordee vs refusee) → a filtrer avant agregation.

> Ces variables qualitatives ne sont pas directement utilisees comme features dans le modele actuel (agregation en comptages numeriques). Si on voulait les integrer, il faudrait un encodage explicite.


## 5. Distributions et outliers

Distribution de chaque feature et detection d'outliers (methode IQR).


In [ ]:
ml_clean = ml[FEATURE_COLUMNS + ALL_TARGETS].dropna()

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=FEATURE_COLUMNS
)

for i, col in enumerate(FEATURE_COLUMNS):
    r, c = divmod(i, 3)
    fig.add_trace(
        go.Box(y=ml_clean[col], name=col, boxpoints='outliers'),
        row=r+1, col=c+1
    )

fig.update_layout(title='Boites a moustaches — features ML (outliers marques)', showlegend=False, height=700)
fig.show()


In [ ]:
print('=== Outliers detectes (methode IQR, seuil 1.5) ===')
for col in FEATURE_COLUMNS:
    s = ml_clean[col]
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((s < Q1 - 1.5*IQR) | (s > Q3 + 1.5*IQR)).sum()
    if n_out > 0:
        print(f'  {col}: {n_out} outlier(s) — min={s.min():.1f}, max={s.max():.1f}')


## 6. Matrice de correlations features → cibles

La correlation de Pearson mesure la relation **lineaire** entre deux variables (entre -1 et +1).


In [ ]:
corr_matrix = ml_clean[FEATURE_COLUMNS + ALL_TARGETS].corr()

fig = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Matrice de correlations (Pearson)',
    text_auto='.2f'
)
fig.show()


In [ ]:
print('=== Correlations features → cibles ===')
for target in ALL_TARGETS:
    print(f'\n-- {target} --')
    corr_t = corr_matrix[target][FEATURE_COLUMNS].sort_values(ascending=False)
    for feat, val in corr_t.items():
        bar = '#' * int(abs(val) * 20)
        sign = '+' if val >= 0 else '-'
        print(f'  {feat:25s} {sign}{bar} ({val:.3f})')


**Lecture :**
- **r > 0.7** : correlation forte positive → la feature croit avec la cible.
- **r < -0.5** : correlation negative moderee a forte.
- **r proche de 0** : pas de relation lineaire (peut exister une relation non lineaire — voir SHAP dans notebook 02).
- Les features tres correlees entre elles (multicollinearite) peuvent poser probleme pour la regression lineaire/Ridge mais pas pour les arbres.


In [ ]:
print('=== Multicollinearite entre features ===')
feat_corr = corr_matrix.loc[FEATURE_COLUMNS, FEATURE_COLUMNS]
pairs = []
for i, f1 in enumerate(FEATURE_COLUMNS):
    for f2 in FEATURE_COLUMNS[i+1:]:
        r = feat_corr.loc[f1, f2]
        if abs(r) > 0.6:
            pairs.append({'feature_1': f1, 'feature_2': f2, 'r': round(r, 3)})

if pairs:
    print('Paires de features fortement correlees (|r| > 0.6) :')
    for p in sorted(pairs, key=lambda x: -abs(x['r'])):
        print(f"  {p['feature_1']:25s} <-> {p['feature_2']:25s}  r={p['r']}")
else:
    print('Pas de multicollinearite forte detectee (|r| <= 0.6 pour toutes les paires).')


## 7. Conclusions et recommandations


### Qualite des donnees
- Verifier ci-dessus le taux de nulls : les colonnes > 30% de manquants ne doivent pas etre utilisees comme features sans imputation.
- Les outliers detectes correspondent generalement a Rennes (commune dominante) — a conserver mais surveiller.

### Variables qualitatives
- `secteur` (public/prive) : non integre comme feature actuellement. Pourrait etre ajoute en ratio `%_prive_par_commune`.
- `type_bien` DVF : utilise uniquement en comptage global (`nb_mutations`). Un filtrage sur les mutations residentielles pourrait affiner le signal.
- `periode_construction` : potentiellement utile pour estimer la composition demographique des batiments.

### Correlations
- Si `population` domine largement les correlations, les autres features apportent de l'information marginale — verifier via SHAP (notebook 02).
- Une forte multicollinearite entre `population` et `densite` est attendue : Ridge regularise cet effet, les arbres l'ignorent naturellement.

### Suite recommandee
- Notebook 02 : explicabilite SHAP pour quantifier la contribution reelle de chaque feature.
- Notebook 03 : benchmark modeles pour confirmer que la complexity vaut le gain.
